# 03 | Gold Layer — Vehicle Sales Analytics

**Project:** US Vehicle Sales Analytics Pipeline  
**Notebook:** `03_gold_analytics`  
**Source:** `workspace.silver.vehicle_monthly_sales`  
**Technology:** Databricks SQL | Delta Lake | Unity Catalog

---

## Objective

Create business-ready analytical tables from validated monthly
passenger-car sales data for consumption by Databricks AI/BI Dashboards.

## Business Requirements

1. Identify the top 10 best-selling passenger-car models by month.
2. Calculate monthly sales totals for manufacturers.
3. Analyze month-over-month sales growth.
4. Calculate model-level sales share and rankings.

## Source Data

**Table:** `workspace.silver.vehicle_monthly_sales`

Columns:
- model
- sales_month
- units_sold
- batch_id
- ingestion_timestamp

## Gold Tables

| Table | Grain | Purpose |
|---|---|---|
| gold.vehicle_monthly_kpis | Model + Month | Model-level monthly KPIs |
| gold.top_10_vehicle_sales | Month + Rank | Top 10 models per month |
| gold.manufacturer_monthly_sales | Manufacturer + Month | Brand performance |
| gold.monthly_sales_trends | Month | Overall sales trends |

## Transformation Logic

- Aggregate valid monthly sales.
- Calculate rankings using ROW_NUMBER.
- Calculate month-over-month growth using LAG.
- Calculate sales shares using window aggregations.
- Exclude months without reported sales data.

## Loading Strategy

Gold tables are rebuilt from the current Silver data using
CREATE OR REPLACE TABLE.

This is a full Gold refresh. Incremental processing is handled
in the Silver layer through Delta MERGE.

For this small dataset, a full Gold refresh is simple and
cost-effective.

## Reporting

Databricks AI/BI Dashboards will consume Gold tables only.

**Scope:** Passenger cars covered by the dataset, not the
entire US vehicle market.


In [0]:
%sql
CREATE OR REPLACE TABLE vehicle_sales.gold.vehicle_monthly_kpis
USING DELTA
AS
WITH base AS (
    SELECT
        model,
        sales_month,
        SUM(units_sold) AS units_sold
    FROM vehicle_sales.silver.vehicle_monthly_sales
    WHERE units_sold IS NOT NULL
      AND units_sold >= 0
    GROUP BY model, sales_month
),
metrics AS (
    SELECT
        model,
        sales_month,
        units_sold,

        SUM(units_sold) OVER (
            PARTITION BY sales_month
        ) AS total_month_sales,

        ROW_NUMBER() OVER (
            PARTITION BY sales_month
            ORDER BY units_sold DESC, model
        ) AS sales_rank

    FROM base
)
SELECT
    m.model,
    m.sales_month,
    m.units_sold,
    m.sales_rank,

    ROUND(
        100.0 * m.units_sold /
        NULLIF(m.total_month_sales, 0),
        2
    ) AS sales_share_pct,

    p.units_sold AS previous_month_sales,

    ROUND(
        100.0 * (
            m.units_sold - p.units_sold
        ) / NULLIF(p.units_sold, 0),
        2
    ) AS mom_growth_pct

FROM metrics m
LEFT JOIN base p
    ON m.model = p.model
   AND p.sales_month = ADD_MONTHS(m.sales_month, -1);

In [0]:
%sql
CREATE OR REPLACE TABLE vehicle_sales.gold.top_10_vehicle_sales
USING DELTA
AS
SELECT
    sales_month,
    sales_rank,
    model,
    units_sold,
    sales_share_pct,
    mom_growth_pct
FROM vehicle_sales.gold.vehicle_monthly_kpis
WHERE sales_rank <= 10;

In [0]:
%sql
SELECT *
FROM vehicle_sales.gold.top_10_vehicle_sales
WHERE sales_month = DATE '2026-09-01'
ORDER BY sales_rank;

In [0]:
%sql
CREATE OR REPLACE TABLE
vehicle_sales.gold.manufacturer_monthly_sales
USING DELTA
AS
WITH brand_sales AS (
    SELECT
        SPLIT(model, ' ')[0] AS manufacturer,
        sales_month,
        SUM(units_sold) AS total_units_sold,
        COUNT(DISTINCT model) AS models_reported

    FROM vehicle_sales.silver.vehicle_monthly_sales
    WHERE units_sold IS NOT NULL
    GROUP BY
        SPLIT(model, ' ')[0],
        sales_month
)
SELECT
    manufacturer,
    sales_month,
    total_units_sold,
    models_reported,

    DENSE_RANK() OVER (
        PARTITION BY sales_month
        ORDER BY total_units_sold DESC
    ) AS manufacturer_rank

FROM brand_sales;

In [0]:
%sql
SELECT *
FROM vehicle_sales.gold.manufacturer_monthly_sales
WHERE sales_month = DATE '2026-09-01'
ORDER BY manufacturer_rank
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE vehicle_sales.gold.monthly_sales_trends
USING DELTA
AS
WITH monthly_totals AS (
    SELECT
        sales_month,
        SUM(units_sold) AS total_units_sold,
        COUNT(DISTINCT model) AS models_reported
    FROM vehicle_sales.silver.vehicle_monthly_sales
    WHERE units_sold IS NOT NULL
    GROUP BY sales_month
)
SELECT
    m.sales_month,
    m.total_units_sold,
    m.models_reported,
    p.total_units_sold AS previous_month_sales,

    ROUND(
        100.0 * (
            m.total_units_sold - p.total_units_sold
        ) / NULLIF(p.total_units_sold, 0),
        2
    ) AS mom_growth_pct

FROM monthly_totals m
LEFT JOIN monthly_totals p
    ON p.sales_month = ADD_MONTHS(m.sales_month, -1);

In [0]:
%sql
SELECT *
FROM vehicle_sales.gold.monthly_sales_trends
ORDER BY sales_month;

In [0]:
%sql
SELECT
    'vehicle_monthly_kpis' AS table_name,
    COUNT(*) AS record_count
FROM vehicle_sales.gold.vehicle_monthly_kpis

UNION ALL

SELECT 'top_10_vehicle_sales', COUNT(*)
FROM vehicle_sales.gold.top_10_vehicle_sales

UNION ALL

SELECT 'manufacturer_monthly_sales', COUNT(*)
FROM vehicle_sales.gold.manufacturer_monthly_sales

UNION ALL

SELECT 'monthly_sales_trends', COUNT(*)
FROM vehicle_sales.gold.monthly_sales_trends;

In [0]:
%sql
WITH silver_totals AS (
    SELECT
        sales_month,
        SUM(units_sold) AS total_sales
    FROM vehicle_sales.silver.vehicle_monthly_sales
    GROUP BY sales_month
)
SELECT
    s.sales_month,
    s.total_sales AS silver_sales,
    g.total_units_sold AS gold_sales,
    s.total_sales - g.total_units_sold AS difference
FROM silver_totals s
JOIN vehicle_sales.gold.monthly_sales_trends g
    ON s.sales_month = g.sales_month
ORDER BY s.sales_month;

In [0]:
%sql
SELECT
    COUNT(DISTINCT model) AS models_reported,
    SUM(units_sold) AS total_sales
FROM vehicle_sales.silver.vehicle_monthly_sales
WHERE sales_month = '2026-09-01';